In [8]:
from google.colab import drive
import os

# Mount Drive
drive.mount('/content/drive')

# Set your directory
DATA_DIR = "/content/drive/MyDrive/Costo de Maternar/Code/"
os.makedirs(DATA_DIR, exist_ok=True)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


# Task
Create a Python script to generate an HTML file for a conjoint survey. The script should define attributes and levels, generate random combinations of these levels for survey questions, and populate an HTML template with these combinations. The final output should be an HTML file containing the survey.

## Define survey structure

### Subtask:
Define the attributes and levels for your conjoint survey.


**Reasoning**:
Create a dictionary to define the attributes and levels for the conjoint survey as specified in the instructions.



In [9]:
import pandas as pd
import random
from dataclasses import dataclass

# ============================================================
# 1. Core inputs
# ============================================================

NOMBRES = {
    "Claudio": "M", "Patricio": "M", "Jorge": "M", "Carlos": "M",
    "Luis": "M", "Daniel": "M",
    "Katherine": "F", "Patricia": "F", "Claudia": "F",
    "Marcela": "F", "Natalia": "F", "Paulina": "F"
}

EDADES = [28, 31, 34, 39]

EDUCACIONES = [
    "Título técnico en Administración de Ventas",
    "Título técnico en Contabilidad",
    "Título técnico en Asistencia Jurídica",
    "título universitario en derecho",
    "título universitario en ingeniería comercial",
    "título universitario en economía",
]

# Cargo mapping — gender handled separately
CARGO_MAP_BASE = {
    "Título técnico en Administración de Ventas": "Encargado de ventas en local comercial",
    "Título técnico en Contabilidad": "Técnico contable",
    "Título técnico en Asistencia Jurídica": "Técnico jurídico",
    "título universitario en derecho": "Abogad@",  # gendered later
    "título universitario en ingeniería comercial": "Analista de ventas",
    "título universitario en economía": "Analista de finanzas",
}

# Children options
HIJOS_OPC = {
    "sin": "no tiene hijos/as",
    "uno_pre": "un hijo/a en preescolar",
    "dos_basica": "2 hijos/as en educación básica",
}

# Marital status base forms
ESTADO_CIVIL_BASE = {
    "solter@": ("soltero", "soltera"),
    "casad@": ("casado", "casada"),
    "conviviente": ("conviviente", "conviviente")
}

INFO = "([INFO])"   # placeholder


# ============================================================
# 2. Gender helpers
# ============================================================

def g_word(genero: str, masc: str, fem: str) -> str:
    return masc if genero == "M" else fem

def articulo(genero: str) -> str:
    return "El" if genero == "M" else "La"

def cargo_from_educacion(educacion: str, genero: str) -> str:
    base = CARGO_MAP_BASE[educacion]
    if base == "Abogad@":
        return g_word(genero, "Abogado", "Abogada")
    return base

def estado_civil_string(key: str, genero: str) -> str:
    masc, fem = ESTADO_CIVIL_BASE[key]
    return masc if genero == "M" else fem


# ============================================================
# 3. Comment templates
# ============================================================

def comentarios_template_A(genero, nombre, hijos_str, estado_str, info=INFO):
    return f"{hijos_str}, {estado_str}, y {info}"

def comentarios_template_B(genero, nombre, hijos_str, estado_str, info=INFO):
    return f"{articulo(genero)} postulante {nombre} está {estado_str}, {hijos_str}; {info}."

TEMPLATES = [comentarios_template_A, comentarios_template_B]

def build_comentarios(genero, nombre, hijos_str, estado_str):
    tmpl = random.choice(TEMPLATES)
    return tmpl(genero, nombre, hijos_str, estado_str, INFO)


# ============================================================
# 4. Experimental condition definitions
# ============================================================

@dataclass
class Condition:
    name: str
    ages: list
    hijos_keys: list
    estado_keys: list
    target_count: int  # number of desired profiles in that cell (not used for filtering full dataset)

CONDICIONES = [
    Condition("sin_hijos_bajo", [28, 31], ["sin"], ["solter@"], 2),
    Condition("sin_hijos_alto", [28, 31, 34], ["sin"], ["casad@", "conviviente"], 4),
    Condition("con_hijos_bajo", [34, 39], ["dos_basica"], ["casad@", "conviviente"], 4),
    Condition("con_hijos_alto", [31, 34], ["uno_pre"], ["casad@", "conviviente"], 4),
]

def matches_condition(edad, hijos_key, estado_key):
    """
    Returns True if the given combo is allowed by ANY experimental condition.
    """
    for cond in CONDICIONES:
        if edad in cond.ages and hijos_key in cond.hijos_keys and estado_key in cond.estado_keys:
            return True
    return False


# ============================================================
# 5. Profile generator
# ============================================================

def generate_profile(nombre, genero, edad, educacion, hijos_key, estado_key):
    hijos_str = HIJOS_OPC[hijos_key]
    estado_str = estado_civil_string(estado_key, genero)
    cargo = cargo_from_educacion(educacion, genero)
    comentarios = build_comentarios(genero, nombre, hijos_str, estado_str)

    return {
        "Name": nombre,
        "Gender": genero,
        "Age": edad,
        "Education": educacion,
        "Current_Role": cargo,
        "Recruiter_Comments": comentarios,
        "Children": hijos_str,
        "Marital_Status": estado_str,
        "Condition": determine_condition(edad, hijos_key, estado_key)
    }


def determine_condition(edad, hijos_key, estado_key):
    for cond in CONDICIONES:
        if edad in cond.ages and hijos_key in cond.hijos_keys and estado_key in cond.estado_keys:
            return cond.name
    return None


# ============================================================
# 6. Generate full dataset of all valid profiles
# ============================================================

def generate_all_valid_profiles():
    profiles = []

    for nombre, genero in NOMBRES.items():
        for edad in EDADES:
            for hijos_key in HIJOS_OPC.keys():
                for estado_key in ESTADO_CIVIL_BASE.keys():
                    # Only proceed if this combo matches at least one experimental condition
                    if not matches_condition(edad, hijos_key, estado_key):
                        continue

                    for educacion in EDUCACIONES:
                        profile = generate_profile(
                            nombre=nombre,
                            genero=genero,
                            edad=edad,
                            educacion=educacion,
                            hijos_key=hijos_key,
                            estado_key=estado_key
                        )
                        profiles.append(profile)

    df = pd.DataFrame(profiles)
    df.insert(0, "id", range(1, len(df) + 1))
    return df


# ============================================================
# 7. Main: Generate and save the master file
# ============================================================

if __name__ == "__main__":
    df_all = generate_all_valid_profiles()
    df_all.to_csv("/content/drive/MyDrive/Costo de Maternar/Code/all_conjoint_profiles.csv",
              index=False, encoding="utf-8")
    print(f"✅ All valid profiles saved ({len(df_all)} profiles) to all_conjoint_profiles.csv")


✅ All valid profiles saved (1152 profiles) to all_conjoint_profiles.csv


In [12]:
import pandas as pd
import numpy as np
import re

# -------------------------------------------------------------
# 1. Load data
# -------------------------------------------------------------
PATH = "/content/drive/MyDrive/Costo de Maternar/Code/all_conjoint_profiles.csv"
df = pd.read_csv(PATH)

print("✅ Loaded dataset with", len(df), "profiles")
print(df.head(), "\n")


# -------------------------------------------------------------
# 2. Basic structure check
# -------------------------------------------------------------
print("=== DF INFO ===")
print(df.info())

print("\n=== BASIC DESCRIPTIVE SUMMARY ===")
print(df.describe(include="all"))


# -------------------------------------------------------------
# 3. Condition counts
# -------------------------------------------------------------
print("\n=== COUNT BY CONDITION ===")
print(df["Condition"].value_counts(), "\n")


# -------------------------------------------------------------
# 4. Gender distribution
# -------------------------------------------------------------
print("=== GENDER DISTRIBUTION ===")
print(df["Gender"].value_counts(), "\n")

print("=== CONDITION × GENDER ===")
print(pd.crosstab(df["Condition"], df["Gender"]), "\n")


# -------------------------------------------------------------
# 5. Age distribution inside each condition
# -------------------------------------------------------------
print("=== CONDITION × AGE ===")
print(pd.crosstab(df["Condition"], df["Age"]), "\n")


# -------------------------------------------------------------
# 6. Marital status distribution
# -------------------------------------------------------------
print("=== CONDITION × MARITAL STATUS ===")
print(pd.crosstab(df["Condition"], df["Marital_Status"]), "\n")


# -------------------------------------------------------------
# 7. Children distribution
# -------------------------------------------------------------
print("=== CONDITION × CHILDREN ===")
print(pd.crosstab(df["Condition"], df["Children"]), "\n")


# -------------------------------------------------------------
# 8. Education distribution
# -------------------------------------------------------------
print("=== CONDITION × EDUCATION ===")
print(pd.crosstab(df["Condition"], df["Education"]), "\n")


# -------------------------------------------------------------
# 9. Full pivot (Condition × Age × Children × Marital × Education)
# -------------------------------------------------------------
print("=== FULL PIVOT (multi-way validation) ===")
pivot = pd.pivot_table(
    df,
    values="id",
    index=["Condition", "Age", "Children", "Marital_Status"],
    columns=["Education"],
    aggfunc="count",
    fill_value=0
)
print(pivot, "\n")


# -------------------------------------------------------------
# 10. Gender agreement checks
# -------------------------------------------------------------

# A) Abogado/a agreement
female_abogado = df[(df["Gender"]=="F") & (df["Current_Role"].str.contains("Abogado"))]
male_abogada = df[(df["Gender"]=="M") & (df["Current_Role"].str.contains("Abogada"))]

print("=== GENDER AGREEMENT CHECK: Abogado/Abogada ===")
print("Female profiles incorrectly labeled 'Abogado':", len(female_abogado))
print("Male profiles incorrectly labeled 'Abogada':", len(male_abogada))

if len(female_abogado) > 0:
    print(female_abogado[["Name","Gender","Current_Role"]])

if len(male_abogada) > 0:
    print(male_abogada[["Name","Gender","Current_Role"]])

print()


# B) Marital status gender agreement: soltero/a & casado/a
wrong_soltero = df[(df["Gender"]=="F") & (df["Marital_Status"].str.contains("soltero"))]
wrong_soltera = df[(df["Gender"]=="M") & (df["Marital_Status"].str.contains("soltera"))]

wrong_casado = df[(df["Gender"]=="F") & (df["Marital_Status"].str.contains("casado"))]
wrong_casada = df[(df["Gender"]=="M") & (df["Marital_Status"].str.contains("casada"))]

print("=== GENDER AGREEMENT CHECK: Soltero/a & Casado/a ===")
print("F but labeled soltero:", len(wrong_soltero))
print("M but labeled soltera:", len(wrong_soltera))
print("F but labeled casado:", len(wrong_casado))
print("M but labeled casada:", len(wrong_casada), "\n")


# -------------------------------------------------------------
# 11. Comment text spot-check
# -------------------------------------------------------------
print("=== SAMPLE COMMENTS (10) ===")
print(df["Recruiter_Comments"].sample(10).tolist(), "\n")


# -------------------------------------------------------------
# 12. Assertions / warnings (light-touch)
# -------------------------------------------------------------
problems = False

# Ensure no missing condition labels
if df["Condition"].isna().any():
    print("❗ WARNING: Some profiles have no assigned experimental condition.")
    problems = True

# Ensure at least some profiles per condition
counts = df["Condition"].value_counts()
if len(counts) < 4:
    print("❗ WARNING: One or more experimental conditions are missing.")
    problems = True

# Ensure INFO placeholder exists
if not df["Recruiter_Comments"].str.contains(r"\[INFO\]", regex=True).any():
    print("❗ WARNING: No INFO placeholder found in comments.")
    problems = True

# Ensure gender mismatches detected above
if len(female_abogado) > 0 or len(male_abogada) > 0:
    print("❗ WARNING: Some lawyer-role gender mismatches detected.")
    problems = True

if len(wrong_soltero) + len(wrong_soltera) + len(wrong_casado) + len(wrong_casada) > 0:
    print("❗ WARNING: Some soltero/soltera or casado/casada mismatches detected.")
    problems = True


# -------------------------------------------------------------
# ✅ Final validation message
# -------------------------------------------------------------
if problems:
    print("\n⚠️ VALIDATION COMPLETE — Some issues were detected (see warnings above).")
else:
    print("\n✅ VALIDATION COMPLETE — Everything looks correct!")

✅ Loaded dataset with 1152 profiles
   id     Name Gender  Age                                     Education  \
0   1  Claudio      M   28    Título técnico en Administración de Ventas   
1   2  Claudio      M   28                Título técnico en Contabilidad   
2   3  Claudio      M   28         Título técnico en Asistencia Jurídica   
3   4  Claudio      M   28               título universitario en derecho   
4   5  Claudio      M   28  título universitario en ingeniería comercial   

                             Current_Role  \
0  Encargado de ventas en local comercial   
1                        Técnico contable   
2                        Técnico jurídico   
3                                 Abogado   
4                      Analista de ventas   

                                  Recruiter_Comments           Children  \
0  El postulante Claudio está soltero, no tiene h...  no tiene hijos/as   
1  El postulante Claudio está soltero, no tiene h...  no tiene hijos/as   
2          